<a href="https://colab.research.google.com/github/Meetra21/Curvature-Orientation-MLP/blob/main/CAML_EMNIST_Confusion_Adaptive_Margin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Confusion-Adaptive Margin Learning (CAML) — EMNIST Letters

This notebook performs the next targeted experiment after the curvature/reliability studies.

A short **pilot model** is trained only on a CORE subset of the official EMNIST training partition. Its errors on a disjoint DISCOVERY subset define the most persistent class-confusion pairs. The final baseline and CAML models are then trained from scratch on the same 90% final-training set, with the same architecture, data order, and deterministic augmentation schedule. The final validation split is separate from pair discovery, and the official test set is used only after model selection.

CAML adds a cosine-space margin only between empirically discovered rival classes; there are no specialists and no post-hoc routing decisions.


In [1]:

# ============================================================
# CONFUSION-ADAPTIVE MARGIN LEARNING (CAML)
# EMNIST Letters
#
# Research question:
# Can a strong augmented CNN reduce its empirically discovered
# class-pair confusions by learning larger embedding margins for
# those specific rival classes?
#
# Protocol:
#  1) Official EMNIST train is split into:
#       - CORE:      pilot training only
#       - DISCOVERY: confusion-graph discovery only
#       - VALIDATION: final model selection only
#  2) A short pilot CNN is trained on CORE.
#  3) Pilot errors on DISCOVERY determine the confusion graph.
#  4) Two final models with IDENTICAL architectures and augmentation:
#       - Baseline: cross-entropy only
#       - CAML: cross-entropy + confusion-adaptive cosine-margin loss
#     are trained on CORE + DISCOVERY.
#  5) Official EMNIST test is evaluated only after model selection.
#
# No pair specialists. No routing. No test-driven pair selection.
# ============================================================

# =========================
# CELL 1 — Dependencies, imports, configuration
# =========================
import sys
import subprocess
import importlib.util

REQUIRED = {
    "torch": "torch",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "tensorflow_datasets": "tensorflow-datasets",
    "importlib_resources": "importlib-resources",
    "tqdm": "tqdm",
}

for module_name, pip_name in REQUIRED.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", pip_name]
        )

import gc
import gzip
import json
import math
import random
import shutil
import struct
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tensorflow_datasets as tfds

from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42

# Official training partition split:
VAL_FRACTION = 0.10
DISCOVERY_FRACTION_OF_FULL_TRAIN = 0.05

# Pilot only discovers the confusion graph; keep it deliberately short.
PILOT_EPOCHS = 5

# Final matched training.
BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512
MAX_EPOCHS = 30
PATIENCE = 6
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
NUM_WORKERS = 0

# Confusion graph.
TOP_CONFUSION_PAIRS = 8
MIN_SYMMETRIC_DISCOVERY_ERRORS = 5

# Proposed loss.
# The cosine-margin is stable because cosine similarities are bounded [-1,1].
BASE_COS_MARGIN = 0.10
EXTRA_COS_MARGIN = 0.10
MARGIN_LOSS_WEIGHT = 0.20

# Same augmentation family that made the previous Raw CNN robust.
# clean, rotation, translation, noise, blur, thicken, thin
AUG_PROBS = [0.30, 0.12, 0.12, 0.12, 0.12, 0.11, 0.11]
assert abs(sum(AUG_PROBS) - 1.0) < 1e-8

# Validation conditions used equally for final checkpoint selection.
VAL_SUITE = [
    ("clean", "clean", 0.0),
    ("rotation_10deg", "rotation", 10.0),
    ("translation_2px", "translation", 2.0),
    ("noise_sigma_0.10", "noise", 0.10),
    ("blur_sigma_1.0", "blur", 1.0),
    ("stroke_thicken_1", "dilate", 1.0),
    ("stroke_thin_1", "erode", 1.0),
]

# Final test suite, fixed before test evaluation.
TEST_SUITE = [
    ("clean", "clean", 0.0),
    ("rotation_5deg", "rotation", 5.0),
    ("rotation_10deg", "rotation", 10.0),
    ("translation_1px", "translation", 1.0),
    ("translation_2px", "translation", 2.0),
    ("noise_sigma_0.05", "noise", 0.05),
    ("noise_sigma_0.10", "noise", 0.10),
    ("blur_sigma_0.6", "blur", 0.6),
    ("blur_sigma_1.0", "blur", 1.0),
    ("stroke_thicken_1", "dilate", 1.0),
    ("stroke_thin_1", "erode", 1.0),
]

LABEL_NAMES = [chr(ord("A") + i) for i in range(26)]

RESULTS_DIR = Path("/content/caml_emnist_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("AMP:", USE_AMP)
print("Results directory:", RESULTS_DIR)


# =========================


Installing missing dependency: importlib-resources
PyTorch: 2.11.0+cu130
Device: cuda
AMP: True
Results directory: /content/caml_emnist_results


In [2]:
# CELL 2 — Load official EMNIST Letters IDX files
# =========================
REQUIRED_FILES = [
    "emnist-letters-train-images-idx3-ubyte.gz",
    "emnist-letters-train-labels-idx1-ubyte.gz",
    "emnist-letters-test-images-idx3-ubyte.gz",
    "emnist-letters-test-labels-idx1-ubyte.gz",
]

TFDS_ROOT = Path("/root/tensorflow_datasets")
DOWNLOAD_DIR = TFDS_ROOT / "downloads"
EXTRACT_DIR = DOWNLOAD_DIR / "extracted"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def find_emnist_file(filename, extra_roots=None):
    roots = []
    if extra_roots:
        roots.extend([Path(x) for x in extra_roots])
    roots.append(TFDS_ROOT)

    for root in roots:
        if root.exists():
            hits = list(root.rglob(filename))
            if hits:
                return hits[0]
    return None

paths = {name: find_emnist_file(name) for name in REQUIRED_FILES}

if any(path is None for path in paths.values()):
    print("Raw EMNIST files not found in cache.")
    print("Downloading the official archive with TFDS DownloadManager...")

    dm = tfds.download.DownloadManager(
        download_dir=str(DOWNLOAD_DIR),
        extract_dir=str(EXTRACT_DIR),
    )

    extracted_path = Path(
        dm.download_and_extract(
            "https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip"
        )
    )

    print("Archive extracted to:", extracted_path)

    paths = {
        name: find_emnist_file(name, extra_roots=[extracted_path])
        for name in REQUIRED_FILES
    }

missing = [name for name, path in paths.items() if path is None]
if missing:
    raise FileNotFoundError(
        "Could not locate these EMNIST files:\n  "
        + "\n  ".join(missing)
    )

def read_idx_images_gz(path):
    with gzip.open(path, "rb") as f:
        header = f.read(16)
        if len(header) != 16:
            raise ValueError(f"Invalid image IDX header: {path}")

        magic, n_images, rows, cols = struct.unpack(">IIII", header)
        if magic != 2051:
            raise ValueError(f"Unexpected image magic {magic}: {path}")

        arr = np.frombuffer(f.read(), dtype=np.uint8)

    expected = n_images * rows * cols
    if arr.size != expected:
        raise ValueError(
            f"Image IDX size mismatch: expected {expected}, found {arr.size}"
        )

    return arr.reshape(n_images, rows, cols)

def read_idx_labels_gz(path):
    with gzip.open(path, "rb") as f:
        header = f.read(8)
        if len(header) != 8:
            raise ValueError(f"Invalid label IDX header: {path}")

        magic, n_labels = struct.unpack(">II", header)
        if magic != 2049:
            raise ValueError(f"Unexpected label magic {magic}: {path}")

        labels = np.frombuffer(f.read(), dtype=np.uint8).astype(np.int64)

    if len(labels) != n_labels:
        raise ValueError(
            f"Label IDX size mismatch: expected {n_labels}, found {len(labels)}"
        )

    return labels

X_all = read_idx_images_gz(
    paths["emnist-letters-train-images-idx3-ubyte.gz"]
)
y_all_raw = read_idx_labels_gz(
    paths["emnist-letters-train-labels-idx1-ubyte.gz"]
)

X_test_u8 = read_idx_images_gz(
    paths["emnist-letters-test-images-idx3-ubyte.gz"]
)
y_test_raw = read_idx_labels_gz(
    paths["emnist-letters-test-labels-idx1-ubyte.gz"]
)

expected_labels = np.arange(1, 27, dtype=np.int64)

if X_all.shape != (124800, 28, 28):
    raise ValueError(f"Expected train shape (124800,28,28), got {X_all.shape}")

if X_test_u8.shape != (20800, 28, 28):
    raise ValueError(f"Expected test shape (20800,28,28), got {X_test_u8.shape}")

if not np.array_equal(np.unique(y_all_raw), expected_labels):
    raise ValueError(f"Unexpected train labels: {np.unique(y_all_raw)}")

if not np.array_equal(np.unique(y_test_raw), expected_labels):
    raise ValueError(f"Unexpected test labels: {np.unique(y_test_raw)}")

# Same convention as prior successful runs: no transpose.
y_all = (y_all_raw - 1).astype(np.int64)
y_test = (y_test_raw - 1).astype(np.int64)

# ------------------------------------------------------------
# Split 1: untouched FINAL validation = 10% of official train.
# ------------------------------------------------------------
idx_all = np.arange(len(y_all))

idx_preval, idx_val = train_test_split(
    idx_all,
    test_size=VAL_FRACTION,
    random_state=SEED,
    stratify=y_all,
)

# ------------------------------------------------------------
# Split 2: discovery = 5% of full official train.
# It is held out from the pilot that discovers the graph.
# ------------------------------------------------------------
n_discovery = int(round(DISCOVERY_FRACTION_OF_FULL_TRAIN * len(y_all)))
discovery_fraction_within_preval = n_discovery / len(idx_preval)

idx_core, idx_discovery = train_test_split(
    idx_preval,
    test_size=discovery_fraction_within_preval,
    random_state=SEED + 1,
    stratify=y_all[idx_preval],
)

X_core_u8 = X_all[idx_core]
y_core = y_all[idx_core]

X_discovery_u8 = X_all[idx_discovery]
y_discovery = y_all[idx_discovery]

X_val_u8 = X_all[idx_val]
y_val = y_all[idx_val]

# Final matched training uses core + discovery = 90% of official train.
idx_final_train = np.concatenate([idx_core, idx_discovery])
X_final_train_u8 = X_all[idx_final_train]
y_final_train = y_all[idx_final_train]

# Integrity: all three official-train subsets are disjoint.
assert len(np.intersect1d(idx_core, idx_discovery)) == 0
assert len(np.intersect1d(idx_core, idx_val)) == 0
assert len(np.intersect1d(idx_discovery, idx_val)) == 0

assert len(X_core_u8) + len(X_discovery_u8) + len(X_val_u8) == 124800
assert len(X_final_train_u8) == len(X_core_u8) + len(X_discovery_u8)

print("\nDATA PROTOCOL")
print("  Pilot CORE         :", X_core_u8.shape, y_core.shape)
print("  DISCOVERY          :", X_discovery_u8.shape, y_discovery.shape)
print("  FINAL VALIDATION   :", X_val_u8.shape, y_val.shape)
print("  FINAL TRAIN        :", X_final_train_u8.shape, y_final_train.shape)
print("  Official TEST      :", X_test_u8.shape, y_test.shape)

# We no longer need full official-train copies/index arrays.
del X_all, y_all, y_all_raw, y_test_raw
del idx_all, idx_preval, idx_final_train
gc.collect()


# =========================


Raw EMNIST files not found in cache.


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Archive extracted to: /root/tensorflow_datasets/downloads/extracted/ZIP.biometrics.nist.gov_cs_links_EMNIST_gzip-5u2fjN3KpzAuJXk7PNtLPNb6LcJaTw1ZM6ioBn82o4.zip

DATA PROTOCOL
  Pilot CORE         : (106080, 28, 28) (106080,)
  DISCOVERY          : (6240, 28, 28) (6240,)
  FINAL VALIDATION   : (12480, 28, 28) (12480,)
  FINAL TRAIN        : (112320, 28, 28) (112320,)
  Official TEST      : (20800, 28, 28) (20800,)


0

In [3]:
# CELL 3 — Dataset / loader helpers
# =========================
class EMNISTRawDataset(Dataset):
    def __init__(self, images_u8, labels):
        self.images = images_u8
        self.labels = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (
            torch.from_numpy(self.images[i][None, :, :]),
            torch.tensor(self.labels[i], dtype=torch.long),
        )

core_ds = EMNISTRawDataset(X_core_u8, y_core)
discovery_ds = EMNISTRawDataset(X_discovery_u8, y_discovery)
final_train_ds = EMNISTRawDataset(X_final_train_u8, y_final_train)
val_ds = EMNISTRawDataset(X_val_u8, y_val)
test_ds = EMNISTRawDataset(X_test_u8, y_test)

def make_loader(
    dataset,
    batch_size,
    shuffle,
    seed=None,
):
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(SEED if seed is None else int(seed))

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

discovery_loader = make_loader(
    discovery_ds,
    EVAL_BATCH_SIZE,
    shuffle=False,
)

val_loader = make_loader(
    val_ds,
    EVAL_BATCH_SIZE,
    shuffle=False,
)

test_loader = make_loader(
    test_ds,
    EVAL_BATCH_SIZE,
    shuffle=False,
)

print("Datasets and loaders ready.")


# =========================


Datasets and loaders ready.


In [4]:
# CELL 4 — Deterministic augmentation / corruption engine
# =========================
def _make_generator(device, seed):
    g = torch.Generator(device=device)
    g.manual_seed(int(seed))
    return g

def _gaussian_kernel2d(sigma, device, dtype):
    radius = 2
    coords = torch.arange(
        -radius,
        radius + 1,
        device=device,
        dtype=dtype,
    )
    k1 = torch.exp(-(coords.square()) / (2.0 * sigma * sigma))
    k1 = k1 / k1.sum()
    k2 = torch.outer(k1, k1)
    return k2[None, None, :, :]

def gaussian_blur_batch(x, sigma):
    k = _gaussian_kernel2d(float(sigma), x.device, x.dtype)
    xp = F.pad(x, (2, 2, 2, 2), mode="reflect")
    return F.conv2d(xp, k)

def apply_condition(x, kind, severity, seed):
    if kind == "clean":
        return x

    B, _, H, W = x.shape
    g = _make_generator(x.device, seed)

    if kind == "rotation":
        angles = (
            torch.rand(B, device=x.device, generator=g) * 2.0 - 1.0
        ) * float(severity) * math.pi / 180.0

        c = torch.cos(angles)
        s = torch.sin(angles)

        theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
        theta[:, 0, 0] = c
        theta[:, 0, 1] = -s
        theta[:, 1, 0] = s
        theta[:, 1, 1] = c

        grid = F.affine_grid(theta, x.size(), align_corners=False)

        return F.grid_sample(
            x,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )

    if kind == "translation":
        sev = int(round(float(severity)))

        dx = torch.randint(
            -sev,
            sev + 1,
            (B,),
            device=x.device,
            generator=g,
        ).to(x.dtype)

        dy = torch.randint(
            -sev,
            sev + 1,
            (B,),
            device=x.device,
            generator=g,
        ).to(x.dtype)

        theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = 2.0 * dx / float(W)
        theta[:, 1, 2] = 2.0 * dy / float(H)

        grid = F.affine_grid(theta, x.size(), align_corners=False)

        return F.grid_sample(
            x,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )

    if kind == "noise":
        noise = torch.randn(
            x.shape,
            device=x.device,
            dtype=x.dtype,
            generator=g,
        ) * float(severity)

        return (x + noise).clamp(0.0, 1.0)

    if kind == "blur":
        return gaussian_blur_batch(x, float(severity))

    if kind == "dilate":
        return F.max_pool2d(
            x,
            kernel_size=3,
            stride=1,
            padding=1,
        )

    if kind == "erode":
        return -F.max_pool2d(
            -x,
            kernel_size=3,
            stride=1,
            padding=1,
        )

    raise ValueError(f"Unknown condition kind: {kind}")

def augment_training_batch(x, seed):
    """
    One corruption family per sample.
    Baseline and CAML receive exactly the same batch order and corruption
    schedule because seeds depend only on epoch/batch, not on the model.
    """
    B, _, H, W = x.shape
    g = _make_generator(x.device, seed)

    probs = torch.tensor(
        AUG_PROBS,
        device=x.device,
        dtype=x.dtype,
    )
    cumulative = torch.cumsum(probs, dim=0)

    u = torch.rand(B, device=x.device, generator=g)
    category = torch.bucketize(u, cumulative[:-1])

    y = x.clone()

    # rotation
    m = category == 1
    if m.any():
        angles = (
            torch.rand(B, device=x.device, generator=g) * 2.0 - 1.0
        ) * 10.0 * math.pi / 180.0

        c = torch.cos(angles)
        s = torch.sin(angles)

        theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
        theta[:, 0, 0] = c
        theta[:, 0, 1] = -s
        theta[:, 1, 0] = s
        theta[:, 1, 1] = c

        grid = F.affine_grid(theta, x.size(), align_corners=False)

        warped = F.grid_sample(
            x,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )

        y[m] = warped[m]

    # translation
    m = category == 2
    if m.any():
        dx = torch.randint(
            -2,
            3,
            (B,),
            device=x.device,
            generator=g,
        ).to(x.dtype)

        dy = torch.randint(
            -2,
            3,
            (B,),
            device=x.device,
            generator=g,
        ).to(x.dtype)

        theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = 2.0 * dx / float(W)
        theta[:, 1, 2] = 2.0 * dy / float(H)

        grid = F.affine_grid(theta, x.size(), align_corners=False)

        shifted = F.grid_sample(
            x,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )

        y[m] = shifted[m]

    # noise sigma uniform [0.02, 0.12]
    m = category == 3
    if m.any():
        sigma = (
            0.02
            + 0.10 * torch.rand(B, device=x.device, generator=g)
        ).to(x.dtype)

        noise = torch.randn(
            x.shape,
            device=x.device,
            dtype=x.dtype,
            generator=g,
        )

        noisy = (
            x
            + noise * sigma[:, None, None, None]
        ).clamp(0.0, 1.0)

        y[m] = noisy[m]

    # blur sigma 0.6 or 1.0
    m = category == 4
    if m.any():
        blur06 = gaussian_blur_batch(x, 0.6)
        blur10 = gaussian_blur_batch(x, 1.0)

        choose10 = (
            torch.rand(B, device=x.device, generator=g)
            >= 0.5
        )

        blurred = torch.where(
            choose10[:, None, None, None],
            blur10,
            blur06,
        )

        y[m] = blurred[m]

    # thicken
    m = category == 5
    if m.any():
        thick = F.max_pool2d(
            x,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        y[m] = thick[m]

    # thin
    m = category == 6
    if m.any():
        thin = -F.max_pool2d(
            -x,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        y[m] = thin[m]

    return y.clamp(0.0, 1.0), category

# Preflight.
with torch.no_grad():
    x0 = torch.rand(8, 1, 28, 28, device=DEVICE)
    xa, cats = augment_training_batch(x0, seed=123)
    assert xa.shape == x0.shape
    assert torch.isfinite(xa).all()

    for _, kind, severity in TEST_SUITE:
        xc = apply_condition(x0, kind, severity, seed=321)
        assert xc.shape == x0.shape
        assert torch.isfinite(xc).all()

print("Augmentation engine preflight passed.")


# =========================


Augmentation engine preflight passed.


In [5]:
# CELL 5 — CNN with explicit 256-D embedding
# Same architecture is used for pilot, baseline, and CAML.
# =========================
def conv_bn_act(cin, cout, k=3, p=1):
    return nn.Sequential(
        nn.Conv2d(
            cin,
            cout,
            kernel_size=k,
            padding=p,
            bias=False,
        ),
        nn.BatchNorm2d(cout),
        nn.GELU(),
    )

class EmbeddingCNN(nn.Module):
    def __init__(self, n_classes=26):
        super().__init__()

        self.raw_stem = nn.Sequential(
            conv_bn_act(1, 32),
            conv_bn_act(32, 32),
        )

        self.adapter = conv_bn_act(32, 32)

        self.trunk = nn.Sequential(
            nn.MaxPool2d(2),
            conv_bn_act(32, 64),
            conv_bn_act(64, 64),
            nn.Dropout2d(0.10),

            nn.MaxPool2d(2),
            conv_bn_act(64, 128),
            conv_bn_act(128, 128),
            nn.Dropout2d(0.15),
        )

        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.GELU(),
        )

        self.embedding_dropout = nn.Dropout(0.35)
        self.classifier = nn.Linear(256, n_classes)

    def forward(self, raw, return_embedding=False):
        x = self.raw_stem(raw)
        x = self.adapter(x)
        x = self.trunk(x)

        emb = self.embedding(x)
        logits = self.classifier(
            self.embedding_dropout(emb)
        )

        if return_embedding:
            return logits, emb

        return logits

def count_trainable(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

probe = EmbeddingCNN().to(DEVICE)

with torch.no_grad():
    z = torch.rand(4, 1, 28, 28, device=DEVICE)
    logits, emb = probe(z, return_embedding=True)

    assert logits.shape == (4, 26)
    assert emb.shape == (4, 256)
    assert torch.isfinite(logits).all()
    assert torch.isfinite(emb).all()

print("Model preflight passed.")
print("Trainable parameters:", f"{count_trainable(probe):,}")

del probe, z, logits, emb

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# =========================


Model preflight passed.
Trainable parameters: 1,908,730


In [6]:
# CELL 6 — Short pilot training on CORE only
# =========================
criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)

def make_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )
    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=USE_AMP
        )

def train_pilot():
    seed_everything(SEED + 100)

    model = EmbeddingCNN().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scaler = make_scaler()

    for epoch in range(1, PILOT_EPOCHS + 1):
        model.train()

        core_loader = make_loader(
            core_ds,
            BATCH_SIZE,
            shuffle=True,
            seed=SEED + 20000 + epoch,
        )

        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (raw_u8, y) in enumerate(
            tqdm(
                core_loader,
                desc=f"Pilot epoch {epoch}",
                leave=False,
            )
        ):
            raw = raw_u8.to(
                DEVICE,
                non_blocking=True,
            ).float().div_(255.0)

            y = y.to(
                DEVICE,
                non_blocking=True,
            )

            raw_aug, _ = augment_training_batch(
                raw,
                seed=700000 + epoch * 10000 + batch_idx,
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP,
            ):
                logits = model(raw_aug)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.item()) * y.size(0)
            correct += int(
                (logits.argmax(dim=1) == y).sum()
            )
            total += int(y.size(0))

        print(
            f"Pilot epoch {epoch:02d} | "
            f"loss={running_loss/total:.4f} | "
            f"train_acc={correct/total:.4%}"
        )

    return model

print("\n" + "=" * 88)
print("SHORT PILOT TRAINING — CORE ONLY")
print("=" * 88)

pilot_model = train_pilot()


# =========================



SHORT PILOT TRAINING — CORE ONLY


Pilot epoch 1:   0%|          | 0/415 [00:00<?, ?it/s]

Pilot epoch 01 | loss=0.7855 | train_acc=86.8467%


Pilot epoch 2:   0%|          | 0/415 [00:00<?, ?it/s]

Pilot epoch 02 | loss=0.5991 | train_acc=92.7555%


...

Pilot epoch 03 | loss=0.5621 | train_acc=93.7849%


Pilot epoch 4:   0%|          | 0/415 [00:00<?, ?it/s]

Pilot epoch 04 | loss=0.5417 | train_acc=94.2553%


Pilot epoch 5:   0%|          | 0/415 [00:00<?, ?it/s]

Pilot epoch 05 | loss=0.5289 | train_acc=94.5711%


In [7]:
# CELL 7 — Discover confusion graph on held-out DISCOVERY split
# =========================
@torch.no_grad()
def predict_clean(model, loader):
    model.eval()

    probs = []
    ys = []

    for raw_u8, y in tqdm(
        loader,
        desc="Discovery inference",
        leave=False,
    ):
        raw = raw_u8.to(
            DEVICE,
            non_blocking=True,
        ).float().div_(255.0)

        logits = model(raw)

        probs.append(
            torch.softmax(
                logits,
                dim=1,
            ).cpu().numpy()
        )

        ys.append(y.numpy())

    return (
        np.concatenate(probs, axis=0),
        np.concatenate(ys, axis=0),
    )

pilot_prob, discovery_y_check = predict_clean(
    pilot_model,
    discovery_loader,
)

if not np.array_equal(discovery_y_check, y_discovery):
    raise RuntimeError("Discovery label ordering mismatch")

pilot_pred = pilot_prob.argmax(axis=1)
pilot_discovery_acc = float(
    np.mean(pilot_pred == y_discovery)
)

cm_discovery = confusion_matrix(
    y_discovery,
    pilot_pred,
    labels=np.arange(26),
)

supports = np.bincount(
    y_discovery,
    minlength=26,
).astype(np.float64)

pair_rows = []

for i in range(26):
    for j in range(i + 1, 26):
        i_to_j = int(cm_discovery[i, j])
        j_to_i = int(cm_discovery[j, i])
        symmetric_errors = i_to_j + j_to_i

        rate_i_j = (
            i_to_j / supports[i]
            if supports[i] > 0
            else 0.0
        )

        rate_j_i = (
            j_to_i / supports[j]
            if supports[j] > 0
            else 0.0
        )

        symmetric_rate = 0.5 * (
            rate_i_j
            + rate_j_i
        )

        pair_rows.append({
            "i": i,
            "j": j,
            "pair": f"{LABEL_NAMES[i]}/{LABEL_NAMES[j]}",
            "i_to_j": i_to_j,
            "j_to_i": j_to_i,
            "symmetric_errors": symmetric_errors,
            "symmetric_rate": symmetric_rate,
        })

pair_df = pd.DataFrame(pair_rows)

eligible = pair_df[
    pair_df["symmetric_errors"]
    >= MIN_SYMMETRIC_DISCOVERY_ERRORS
].copy()

eligible = eligible.sort_values(
    ["symmetric_rate", "symmetric_errors"],
    ascending=False,
).reset_index(drop=True)

if len(eligible) < TOP_CONFUSION_PAIRS:
    raise RuntimeError(
        f"Only {len(eligible)} eligible confusion pairs were found; "
        f"need {TOP_CONFUSION_PAIRS}. "
        "Lower MIN_SYMMETRIC_DISCOVERY_ERRORS if necessary."
    )

selected_pairs_df = eligible.head(
    TOP_CONFUSION_PAIRS
).copy()

max_rate = float(
    selected_pairs_df["symmetric_rate"].max()
)

if max_rate <= 0:
    raise RuntimeError("Confusion graph has zero pair rates")

selected_pairs_df["weight"] = (
    selected_pairs_df["symmetric_rate"]
    / max_rate
)

selected_pairs_df.to_csv(
    RESULTS_DIR / "discovered_confusion_pairs.csv",
    index=False,
)

print("\n" + "=" * 88)
print("CONFUSION GRAPH DISCOVERED WITHOUT USING FINAL VALIDATION OR TEST")
print("=" * 88)
print(f"Pilot discovery accuracy: {pilot_discovery_acc:.4%}")
print(
    selected_pairs_df[
        [
            "pair",
            "i_to_j",
            "j_to_i",
            "symmetric_errors",
            "symmetric_rate",
            "weight",
        ]
    ].to_string(index=False)
)

# Symmetric 26x26 graph-weight matrix.
W_np = np.zeros((26, 26), dtype=np.float32)

for row in selected_pairs_df.itertuples():
    W_np[row.i, row.j] = float(row.weight)
    W_np[row.j, row.i] = float(row.weight)

W = torch.tensor(
    W_np,
    dtype=torch.float32,
    device=DEVICE,
)

selected_pair_set = {
    tuple(sorted((int(row.i), int(row.j))))
    for row in selected_pairs_df.itertuples()
}

# Pilot is no longer needed.
del pilot_model, pilot_prob, pilot_pred
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()


# =========================


Discovery inference:   0%|          | 0/13 [00:00<?, ?it/s]


CONFUSION GRAPH DISCOVERED WITHOUT USING FINAL VALIDATION OR TEST
Pilot discovery accuracy: 95.3365%
pair  i_to_j  j_to_i  symmetric_errors  symmetric_rate   weight
 I/L      38      66               104        0.216667 1.000000
 G/Q      24      14                38        0.079167 0.365385
 U/V       3       6                 9        0.018750 0.086538
 R/V       8       1                 9        0.018750 0.086538
 D/O       7       1                 8        0.016667 0.076923
 C/E       4       3                 7        0.014583 0.067308
 I/J       3       4                 7        0.014583 0.067308
 H/N       4       2                 6        0.012500 0.057692


213

In [8]:
# CELL 8 — Confusion-adaptive cosine-margin loss
# =========================
def confusion_margin_loss(
    embeddings,
    labels,
    classifier_weight,
    graph_weights,
):
    """
    For each sample y and discovered rival j:
        sim(y) - sim(j) >= adaptive_margin(y,j)

    sim() is cosine similarity between the normalized embedding and
    normalized classifier class weight. The margin therefore does not
    depend on the arbitrary scale of cross-entropy logits.

    Returns:
        scalar loss,
        fraction of active pair constraints violated.
    """
    emb_n = F.normalize(
        embeddings.float(),
        p=2,
        dim=1,
    )

    cls_n = F.normalize(
        classifier_weight.float(),
        p=2,
        dim=1,
    )

    cosine = emb_n @ cls_n.T

    batch_idx = torch.arange(
        labels.size(0),
        device=labels.device,
    )

    true_sim = cosine[
        batch_idx,
        labels,
    ].unsqueeze(1)

    weights = graph_weights[labels]  # [B,26]
    mask = weights > 0

    if not mask.any():
        return cosine.new_zeros(()), 0.0

    adaptive_margin = (
        BASE_COS_MARGIN
        + EXTRA_COS_MARGIN * weights
    )

    violation = (
        adaptive_margin
        - (true_sim - cosine)
    )

    # Hinge only on discovered rivals.
    hinge = F.relu(violation) * weights

    denom = weights[mask].sum().clamp_min(1e-8)
    loss = hinge[mask].sum() / denom

    violated_fraction = float(
        (violation[mask] > 0)
        .float()
        .mean()
        .detach()
        .cpu()
    )

    return loss, violated_fraction

# Synthetic numerical preflight, including backward pass.
probe = EmbeddingCNN().to(DEVICE)
probe.train()

xb = torch.rand(16, 1, 28, 28, device=DEVICE)
yb = torch.arange(16, device=DEVICE) % 26

logits_b, emb_b = probe(
    xb,
    return_embedding=True,
)

ce_b = criterion(
    logits_b,
    yb,
)

margin_b, violated_b = confusion_margin_loss(
    emb_b,
    yb,
    probe.classifier.weight,
    W,
)

total_b = (
    ce_b
    + MARGIN_LOSS_WEIGHT * margin_b
)

if not torch.isfinite(total_b):
    raise RuntimeError("Non-finite CAML preflight loss")

total_b.backward()

print(
    "CAML loss preflight passed | "
    f"CE={ce_b.item():.4f}, "
    f"margin={margin_b.item():.4f}, "
    f"violated={violated_b:.3f}"
)

del probe, xb, yb, logits_b, emb_b, ce_b, margin_b, total_b
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# =========================


CAML loss preflight passed | CE=3.3101, margin=0.1703, violated=1.000


In [9]:
# CELL 9 — Final training/evaluation helpers
# =========================
@torch.no_grad()
def evaluate_condition(
    model,
    loader,
    kind,
    severity,
    seed_base,
):
    model.eval()

    correct = 0
    total = 0

    for batch_idx, (raw_u8, y) in enumerate(loader):
        raw = raw_u8.to(
            DEVICE,
            non_blocking=True,
        ).float().div_(255.0)

        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        raw = apply_condition(
            raw,
            kind,
            severity,
            seed=seed_base + batch_idx,
        )

        logits = model(raw)
        pred = logits.argmax(dim=1)

        correct += int(
            (pred == y).sum()
        )
        total += int(y.numel())

    return correct / total

def evaluate_validation_suite(model):
    rows = []

    for cond_index, (name, kind, severity) in enumerate(VAL_SUITE):
        acc = evaluate_condition(
            model,
            val_loader,
            kind,
            severity,
            seed_base=500000 + cond_index * 10000,
        )

        rows.append({
            "condition": name,
            "accuracy": float(acc),
        })

    df = pd.DataFrame(rows)

    score = float(
        df["accuracy"].mean()
    )

    clean_acc = float(
        df.loc[
            df["condition"] == "clean",
            "accuracy",
        ].iloc[0]
    )

    return score, clean_acc, df

def train_final_model(
    name,
    use_caml,
):
    seed_everything(SEED)

    model = EmbeddingCNN().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )

    scaler = make_scaler()

    best_score = -np.inf
    best_state = None
    best_epoch = None
    bad_epochs = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        train_loader = make_loader(
            final_train_ds,
            BATCH_SIZE,
            shuffle=True,
            seed=SEED + 10000 + epoch,
        )

        running_total_loss = 0.0
        running_ce_loss = 0.0
        running_margin_loss = 0.0
        violated_sum = 0.0
        violated_batches = 0

        correct = 0
        total = 0

        for batch_idx, (raw_u8, y) in enumerate(
            tqdm(
                train_loader,
                desc=f"{name} epoch {epoch}",
                leave=False,
            )
        ):
            raw = raw_u8.to(
                DEVICE,
                non_blocking=True,
            ).float().div_(255.0)

            y = y.to(
                DEVICE,
                non_blocking=True,
            )

            # Exactly the same deterministic augmented input for
            # baseline and CAML at the same epoch/batch index.
            raw_aug, _ = augment_training_batch(
                raw,
                seed=900000 + epoch * 10000 + batch_idx,
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP,
            ):
                logits, emb = model(
                    raw_aug,
                    return_embedding=True,
                )

                ce_loss = criterion(
                    logits,
                    y,
                )

                if use_caml:
                    margin_loss, violated_fraction = confusion_margin_loss(
                        emb,
                        y,
                        model.classifier.weight,
                        W,
                    )
                    total_loss = (
                        ce_loss
                        + MARGIN_LOSS_WEIGHT * margin_loss
                    )
                else:
                    margin_loss = ce_loss.new_zeros(())
                    violated_fraction = 0.0
                    total_loss = ce_loss

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs = int(y.size(0))

            running_total_loss += float(total_loss.item()) * bs
            running_ce_loss += float(ce_loss.item()) * bs
            running_margin_loss += float(margin_loss.item()) * bs

            if use_caml:
                violated_sum += violated_fraction
                violated_batches += 1

            correct += int(
                (logits.argmax(dim=1) == y).sum()
            )
            total += bs

        train_acc = correct / total

        val_score, val_clean_acc, val_df = evaluate_validation_suite(
            model
        )

        scheduler.step(val_score)
        lr_now = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch,
            "train_total_loss": running_total_loss / total,
            "train_ce_loss": running_ce_loss / total,
            "train_margin_loss": running_margin_loss / total,
            "train_accuracy": train_acc,
            "mean_constraint_violation_fraction": (
                violated_sum / violated_batches
                if violated_batches > 0
                else 0.0
            ),
            "val_clean_accuracy": val_clean_acc,
            "val_robust_mean_accuracy": val_score,
            "lr": lr_now,
        })

        print(
            f"{name} | epoch {epoch:02d} | "
            f"loss={running_total_loss/total:.4f} | "
            f"ce={running_ce_loss/total:.4f} | "
            f"margin={running_margin_loss/total:.4f} | "
            f"train={train_acc:.4%} | "
            f"val_clean={val_clean_acc:.4%} | "
            f"val_mean={val_score:.4%} | "
            f"lr={lr_now:.2e}"
        )

        if val_score > best_score + 1e-5:
            best_score = val_score
            best_epoch = epoch

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            bad_epochs = 0

            val_df.to_csv(
                RESULTS_DIR / f"{name}_best_validation_suite.csv",
                index=False,
            )

        else:
            bad_epochs += 1

            if bad_epochs >= PATIENCE:
                print(
                    f"Early stopping {name}; "
                    f"best epoch={best_epoch}, "
                    f"best val mean={best_score:.4%}"
                )
                break

    if best_state is None:
        raise RuntimeError(
            f"No best checkpoint captured for {name}"
        )

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        RESULTS_DIR / f"{name}_history.csv",
        index=False,
    )

    torch.save(
        model.state_dict(),
        RESULTS_DIR / f"{name}_best.pt",
    )

    return (
        model,
        float(best_score),
        int(best_epoch),
        history_df,
    )


# =========================


In [10]:
# CELL 10 — Train matched CE baseline
# =========================
print("\n" + "=" * 88)
print("FINAL MODEL 1/2 — MATCHED AUGMENTED CE BASELINE")
print("=" * 88)

baseline_model, baseline_best_val, baseline_best_epoch, baseline_history = (
    train_final_model(
        "baseline_ce",
        use_caml=False,
    )
)

print(
    f"Baseline complete: "
    f"best epoch={baseline_best_epoch}, "
    f"best val mean={baseline_best_val:.4%}"
)


# =========================



FINAL MODEL 1/2 — MATCHED AUGMENTED CE BASELINE


baseline_ce epoch 1:   0%|          | 0/439 [00:00<?, ?it/s]

baseline_ce | epoch 01 | loss=0.7848 | ce=0.7848 | margin=0.0000 | train=86.8750% | val_clean=93.9984% | val_mean=92.7770% | lr=1.00e-03


baseline_ce epoch 2:   0%|          | 0/439 [00:00<?, ?it/s]

baseline_ce | epoch 02 | loss=0.5940 | ce=0.5940 | margin=0.0000 | train=93.0075% | val_clean=94.5994% | val_mean=93.7214% | lr=1.00e-03


...

baseline_ce | epoch 28 | loss=0.4383 | ce=0.4383 | margin=0.0000 | train=97.3985% | val_clean=95.7372% | val_mean=95.4785% | lr=6.25e-05


baseline_ce epoch 29:   0%|          | 0/439 [00:00<?, ?it/s]

baseline_ce | epoch 29 | loss=0.4373 | ce=0.4373 | margin=0.0000 | train=97.3762% | val_clean=95.7372% | val_mean=95.4224% | lr=6.25e-05


baseline_ce epoch 30:   0%|          | 0/439 [00:00<?, ?it/s]

baseline_ce | epoch 30 | loss=0.4367 | ce=0.4367 | margin=0.0000 | train=97.4430% | val_clean=95.7452% | val_mean=95.4407% | lr=3.13e-05
Early stopping baseline_ce; best epoch=24, best val mean=95.5151%
Baseline complete: best epoch=24, best val mean=95.5151%


In [11]:
# CELL 11 — Train CAML proposed model
# =========================
print("\n" + "=" * 88)
print("FINAL MODEL 2/2 — CONFUSION-ADAPTIVE MARGIN LEARNING")
print("=" * 88)

caml_model, caml_best_val, caml_best_epoch, caml_history = (
    train_final_model(
        "caml",
        use_caml=True,
    )
)

print(
    f"CAML complete: "
    f"best epoch={caml_best_epoch}, "
    f"best val mean={caml_best_val:.4%}"
)


# =========================



FINAL MODEL 2/2 — CONFUSION-ADAPTIVE MARGIN LEARNING


caml epoch 1:   0%|          | 0/439 [00:00<?, ?it/s]

caml | epoch 01 | loss=0.8048 | ce=0.7886 | margin=0.0813 | train=86.8412% | val_clean=93.9744% | val_mean=92.7747% | lr=1.00e-03


caml epoch 2:   0%|          | 0/439 [00:00<?, ?it/s]

caml | epoch 02 | loss=0.6099 | ce=0.5959 | margin=0.0697 | train=92.9647% | val_clean=94.5192% | val_mean=93.7065% | lr=1.00e-03


...

caml | epoch 28 | loss=0.4475 | ce=0.4375 | margin=0.0504 | train=97.4190% | val_clean=95.7612% | val_mean=95.4235% | lr=1.25e-04


caml epoch 29:   0%|          | 0/439 [00:00<?, ?it/s]

caml | epoch 29 | loss=0.4471 | ce=0.4370 | margin=0.0506 | train=97.4484% | val_clean=95.6891% | val_mean=95.4258% | lr=1.25e-04


caml epoch 30:   0%|          | 0/439 [00:00<?, ?it/s]

caml | epoch 30 | loss=0.4469 | ce=0.4368 | margin=0.0506 | train=97.4181% | val_clean=95.7772% | val_mean=95.4533% | lr=6.25e-05
CAML complete: best epoch=27, best val mean=95.5117%


In [12]:
# CELL 12 — Final official-test benchmark
# =========================
@torch.no_grad()
def predict_condition(
    model,
    loader,
    kind,
    severity,
    seed_base,
):
    model.eval()

    probs = []
    embeddings = []
    ys = []

    for batch_idx, (raw_u8, y) in enumerate(loader):
        raw = raw_u8.to(
            DEVICE,
            non_blocking=True,
        ).float().div_(255.0)

        raw = apply_condition(
            raw,
            kind,
            severity,
            seed=seed_base + batch_idx,
        )

        logits, emb = model(
            raw,
            return_embedding=True,
        )

        probs.append(
            torch.softmax(
                logits,
                dim=1,
            ).float().cpu().numpy()
        )

        embeddings.append(
            emb.float().cpu().numpy()
        )

        ys.append(
            y.numpy()
        )

    return (
        np.concatenate(probs, axis=0),
        np.concatenate(embeddings, axis=0),
        np.concatenate(ys, axis=0),
    )

def exact_mcnemar(
    y,
    pred_base,
    pred_caml,
):
    base_correct = pred_base == y
    caml_correct = pred_caml == y

    n01 = int(
        (
            ~base_correct
            & caml_correct
        ).sum()
    )

    n10 = int(
        (
            base_correct
            & ~caml_correct
        ).sum()
    )

    n = n01 + n10

    p = (
        float(
            binomtest(
                min(n01, n10),
                n=n,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )
        if n > 0
        else 1.0
    )

    return n01, n10, p

test_rows = []

clean_base_pred = None
clean_caml_pred = None
clean_base_prob = None
clean_caml_prob = None
clean_y = None

for cond_index, (
    condition,
    kind,
    severity,
) in enumerate(TEST_SUITE):

    print("\n" + "=" * 80)
    print("TEST CONDITION:", condition)
    print("=" * 80)

    seed_base = (
        800000
        + cond_index * 10000
    )

    base_prob, _, y1 = predict_condition(
        baseline_model,
        test_loader,
        kind,
        severity,
        seed_base=seed_base,
    )

    caml_prob, _, y2 = predict_condition(
        caml_model,
        test_loader,
        kind,
        severity,
        seed_base=seed_base,
    )

    if not np.array_equal(y1, y_test):
        raise RuntimeError(
            "Baseline test label ordering mismatch"
        )

    if not np.array_equal(y2, y_test):
        raise RuntimeError(
            "CAML test label ordering mismatch"
        )

    base_pred = base_prob.argmax(axis=1)
    caml_pred = caml_prob.argmax(axis=1)

    base_acc = float(
        np.mean(base_pred == y_test)
    )

    caml_acc = float(
        np.mean(caml_pred == y_test)
    )

    n01, n10, p = exact_mcnemar(
        y_test,
        base_pred,
        caml_pred,
    )

    if condition == "clean":
        clean_base_pred = base_pred.copy()
        clean_caml_pred = caml_pred.copy()
        clean_base_prob = base_prob.copy()
        clean_caml_prob = caml_prob.copy()
        clean_y = y_test.copy()

    test_rows.append({
        "condition": condition,
        "corruption": kind,
        "severity": severity,
        "baseline_accuracy": base_acc,
        "caml_accuracy": caml_acc,
        "caml_minus_baseline_points": 100.0 * (
            caml_acc - base_acc
        ),
        "baseline_wrong_caml_correct": n01,
        "baseline_correct_caml_wrong": n10,
        "mcnemar_p_value": p,
    })

    print(
        f"Baseline accuracy : {base_acc*100:.4f}%"
    )
    print(
        f"CAML accuracy     : {caml_acc*100:.4f}%"
    )
    print(
        f"CAML - Baseline   : "
        f"{(caml_acc-base_acc)*100:+.4f} points"
    )
    print(
        f"McNemar fixed/broken: "
        f"{n01}/{n10}, p={p:.8g}"
    )

    del base_prob, caml_prob, base_pred, caml_pred
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

test_df = pd.DataFrame(test_rows)


# =========================



TEST CONDITION: clean


/tmp/ipykernel_2809/3765116443.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(self.images[i][None, :, :]),


Baseline accuracy : 95.6346%
CAML accuracy     : 95.6731%
CAML - Baseline   : +0.0385 points
McNemar fixed/broken: 124/116, p=0.65146909

TEST CONDITION: rotation_5deg
Baseline accuracy : 95.5240%
CAML accuracy     : 95.6298%
CAML - Baseline   : +0.1058 points
McNemar fixed/broken: 133/111, p=0.17870383

TEST CONDITION: rotation_10deg
Baseline accuracy : 95.5144%
CAML accuracy     : 95.5433%
CAML - Baseline   : +0.0288 points
McNemar fixed/broken: 134/128, p=0.75746483

TEST CONDITION: translation_1px
Baseline accuracy : 95.4327%
CAML accuracy     : 95.5337%
CAML - Baseline   : +0.1010 points
McNemar fixed/broken: 125/104, p=0.18616989

TEST CONDITION: translation_2px
Baseline accuracy : 95.3269%
CAML accuracy     : 95.3269%
CAML - Baseline   : +0.0000 points
McNemar fixed/broken: 141/141, p=1

TEST CONDITION: noise_sigma_0.05
Baseline accuracy : 95.4663%
CAML accuracy     : 95.5529%
CAML - Baseline   : +0.0865 points
McNemar fixed/broken: 127/109, p=0.26842731

TEST CONDITION: noise_s

In [13]:
# CELL 13 — Targeted pair analysis + statistics + summary
# =========================
# Holm family-wise correction over non-clean robustness comparisons.
nonclean_mask = test_df["condition"] != "clean"

pvals = test_df.loc[
    nonclean_mask,
    "mcnemar_p_value",
].to_numpy(dtype=float)

m = len(pvals)
order = np.argsort(pvals)
adjusted = np.empty(m, dtype=float)

running_max = 0.0

for rank, original_idx in enumerate(order):
    candidate = min(
        1.0,
        (m - rank) * pvals[original_idx],
    )

    running_max = max(
        running_max,
        candidate,
    )

    adjusted[original_idx] = running_max

test_df["holm_adjusted_p"] = np.nan
test_df.loc[
    nonclean_mask,
    "holm_adjusted_p",
] = adjusted

test_df.to_csv(
    RESULTS_DIR / "test_robustness_baseline_vs_caml.csv",
    index=False,
)

# Clean macro F1.
def macro_f1(y, pred):
    return float(
        precision_recall_fscore_support(
            y,
            pred,
            average="macro",
            zero_division=0,
        )[2]
    )

baseline_clean_f1 = macro_f1(
    clean_y,
    clean_base_pred,
)

caml_clean_f1 = macro_f1(
    clean_y,
    clean_caml_pred,
)

# Clean confusion matrices.
cm_base = confusion_matrix(
    clean_y,
    clean_base_pred,
    labels=np.arange(26),
)

cm_caml = confusion_matrix(
    clean_y,
    clean_caml_pred,
    labels=np.arange(26),
)

selected_pair_rows = []

for row in selected_pairs_df.itertuples():
    i = int(row.i)
    j = int(row.j)

    base_errors = int(
        cm_base[i, j]
        + cm_base[j, i]
    )

    caml_errors = int(
        cm_caml[i, j]
        + cm_caml[j, i]
    )

    selected_pair_rows.append({
        "pair": f"{LABEL_NAMES[i]}/{LABEL_NAMES[j]}",
        "discovery_weight": float(row.weight),
        "baseline_test_symmetric_errors": base_errors,
        "caml_test_symmetric_errors": caml_errors,
        "error_change": caml_errors - base_errors,
        "error_reduction_fraction": (
            (base_errors - caml_errors) / base_errors
            if base_errors > 0
            else 0.0
        ),
    })

selected_pair_test_df = pd.DataFrame(
    selected_pair_rows
).sort_values(
    "baseline_test_symmetric_errors",
    ascending=False,
)

selected_pair_test_df.to_csv(
    RESULTS_DIR / "selected_pair_clean_test_analysis.csv",
    index=False,
)

base_selected_total = int(
    selected_pair_test_df[
        "baseline_test_symmetric_errors"
    ].sum()
)

caml_selected_total = int(
    selected_pair_test_df[
        "caml_test_symmetric_errors"
    ].sum()
)

selected_pair_net_reduction = (
    (base_selected_total - caml_selected_total)
    / base_selected_total
    if base_selected_total > 0
    else 0.0
)

# Overall means.
corrupted = test_df[nonclean_mask].copy()

mean_base = float(
    corrupted["baseline_accuracy"].mean()
)

mean_caml = float(
    corrupted["caml_accuracy"].mean()
)

wins = int(
    (
        corrupted["caml_accuracy"]
        > corrupted["baseline_accuracy"]
    ).sum()
)

ties = int(
    (
        corrupted["caml_accuracy"]
        == corrupted["baseline_accuracy"]
    ).sum()
)

losses = int(
    (
        corrupted["caml_accuracy"]
        < corrupted["baseline_accuracy"]
    ).sum()
)

baseline_clean_acc = float(
    test_df.loc[
        test_df.condition == "clean",
        "baseline_accuracy",
    ].iloc[0]
)

caml_clean_acc = float(
    test_df.loc[
        test_df.condition == "clean",
        "caml_accuracy",
    ].iloc[0]
)

summary = {
    "seed": SEED,
    "core_size": int(len(y_core)),
    "discovery_size": int(len(y_discovery)),
    "final_train_size": int(len(y_final_train)),
    "validation_size": int(len(y_val)),
    "test_size": int(len(y_test)),
    "pilot_epochs": int(PILOT_EPOCHS),
    "pilot_discovery_accuracy": float(pilot_discovery_acc),
    "selected_pairs": selected_pairs_df["pair"].tolist(),
    "margin_loss_weight": float(MARGIN_LOSS_WEIGHT),
    "base_cos_margin": float(BASE_COS_MARGIN),
    "extra_cos_margin": float(EXTRA_COS_MARGIN),
    "baseline_best_validation_mean": float(baseline_best_val),
    "caml_best_validation_mean": float(caml_best_val),
    "baseline_best_epoch": int(baseline_best_epoch),
    "caml_best_epoch": int(caml_best_epoch),
    "baseline_clean_accuracy": baseline_clean_acc,
    "caml_clean_accuracy": caml_clean_acc,
    "clean_accuracy_gain_points": 100.0 * (
        caml_clean_acc - baseline_clean_acc
    ),
    "baseline_clean_macro_f1": baseline_clean_f1,
    "caml_clean_macro_f1": caml_clean_f1,
    "baseline_mean_corrupted_accuracy": mean_base,
    "caml_mean_corrupted_accuracy": mean_caml,
    "mean_corrupted_gain_points": 100.0 * (
        mean_caml - mean_base
    ),
    "selected_pair_baseline_errors": base_selected_total,
    "selected_pair_caml_errors": caml_selected_total,
    "selected_pair_error_reduction_fraction": float(
        selected_pair_net_reduction
    ),
    "caml_wins": wins,
    "ties": ties,
    "caml_losses": losses,
    "trainable_parameters": int(
        count_trainable(caml_model)
    ),
}

with open(
    RESULTS_DIR / "summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

print("\n" + "=" * 112)
print("CONFUSION-ADAPTIVE MARGIN LEARNING — RESULTS TO SEND BACK")
print("=" * 112)

print("\nDISCOVERED PAIRS")
print(
    selected_pairs_df[
        [
            "pair",
            "symmetric_errors",
            "symmetric_rate",
            "weight",
        ]
    ].to_string(index=False)
)

print("\nFINAL TEST RESULTS")
print(
    test_df[
        [
            "condition",
            "baseline_accuracy",
            "caml_accuracy",
            "caml_minus_baseline_points",
            "baseline_wrong_caml_correct",
            "baseline_correct_caml_wrong",
            "mcnemar_p_value",
            "holm_adjusted_p",
        ]
    ].to_string(index=False)
)

print("\nSELECTED-PAIR CLEAN TEST ANALYSIS")
print(
    selected_pair_test_df.to_string(
        index=False
    )
)

print()
print(f"Baseline clean accuracy             : {baseline_clean_acc*100:.4f}%")
print(f"CAML clean accuracy                 : {caml_clean_acc*100:.4f}%")
print(f"Clean CAML - Baseline               : {(caml_clean_acc-baseline_clean_acc)*100:+.4f} points")
print(f"Baseline clean macro F1             : {baseline_clean_f1:.6f}")
print(f"CAML clean macro F1                 : {caml_clean_f1:.6f}")
print()
print(f"Mean corrupted accuracy — Baseline  : {mean_base*100:.4f}%")
print(f"Mean corrupted accuracy — CAML      : {mean_caml*100:.4f}%")
print(f"Mean corrupted CAML - Baseline      : {(mean_caml-mean_base)*100:+.4f} points")
print(f"CAML wins / ties / losses           : {wins} / {ties} / {losses}")
print()
print(f"Selected-pair errors — Baseline     : {base_selected_total}")
print(f"Selected-pair errors — CAML         : {caml_selected_total}")
print(f"Selected-pair error reduction       : {selected_pair_net_reduction*100:+.4f}%")
print()
print(f"Best validation mean — Baseline     : {baseline_best_val*100:.4f}% (epoch {baseline_best_epoch})")
print(f"Best validation mean — CAML         : {caml_best_val*100:.4f}% (epoch {caml_best_epoch})")
print(f"Pilot discovery accuracy            : {pilot_discovery_acc*100:.4f}%")
print(f"Trainable parameters/model          : {count_trainable(caml_model):,}")
print("=" * 112)
print("Send this complete block back to ChatGPT.")
print("=" * 112)


# =========================



CONFUSION-ADAPTIVE MARGIN LEARNING — RESULTS TO SEND BACK

DISCOVERED PAIRS
pair  symmetric_errors  symmetric_rate   weight
 I/L               104        0.216667 1.000000
 G/Q                38        0.079167 0.365385
 U/V                 9        0.018750 0.086538
 R/V                 9        0.018750 0.086538
 D/O                 8        0.016667 0.076923
 C/E                 7        0.014583 0.067308
 I/J                 7        0.014583 0.067308
 H/N                 6        0.012500 0.057692

FINAL TEST RESULTS
       condition  baseline_accuracy  caml_accuracy  caml_minus_baseline_points  baseline_wrong_caml_correct  baseline_correct_caml_wrong  mcnemar_p_value  holm_adjusted_p
           clean           0.956346       0.956731                    0.038462                          124                          116         0.651469              NaN
   rotation_5deg           0.955240       0.956298                    0.105769                          133                      

In [14]:
# CELL 14 — Save/download all artifacts
# =========================
zip_path = shutil.make_archive(
    "/content/caml_emnist_results",
    "zip",
    root_dir=str(RESULTS_DIR),
)

print("Results archive:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download skipped:", repr(e))


Results archive: /content/caml_emnist_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>